# Phase 35 — LSTM (Long Short-Term Memory)

Υλοποίηση BiLSTM για Food Hazard Detection.

**Αρχιτεκτονική:**
```
Text → Embedding → BiLSTM → Dropout → Linear → Output
```

**Γιατί BiLSTM:**
- Διαβάζει το κείμενο και προς τις δύο κατευθύνσεις
- Μαθαίνει long-range dependencies (σε αντίθεση με CNN)

**Διαφορά από TextCNN:**
- CNN: μαθαίνει τοπικά patterns (n-grams)
- LSTM: μαθαίνει sequential dependencies

**Διαφορά από BERT:**
- BERT: bidirectional transformer με attention
- LSTM: sequential, χωρίς attention mechanism

**Σκοπός:** Σύγκριση CNN vs LSTM vs BERT για την αναφορά

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import re
import copy
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from collections import Counter
import matplotlib.pyplot as plt
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
all_hazard  = pd.concat([train['hazard-category'],  valid['hazard-category']])
all_product = pd.concat([train['product-category'], valid['product-category']])
le_hazard.fit(all_hazard)
le_product.fit(all_product)

y_train_hazard  = le_hazard.transform(train['hazard-category'])
y_train_product = le_product.transform(train['product-category'])
y_valid_hazard  = le_hazard.transform(valid['hazard-category'])
y_valid_product = le_product.transform(valid['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')
print(f'Hazard: {n_hazard} classes | Product: {n_product} classes')

In [ ]:
# Vocabulary
MAX_VOCAB = 30000
MAX_LEN   = 128
PAD_IDX   = 0
UNK_IDX   = 1

def tokenize(text):
    return text.lower().split()

counter = Counter()
for text in texts_train:
    counter.update(tokenize(text))

vocab    = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

print(f'Vocabulary size: {len(vocab):,}')

def text_to_ids(text, max_len=MAX_LEN):
    tokens = tokenize(text)[:max_len]
    ids    = [vocab.get(t, UNK_IDX) for t in tokens]
    ids   += [PAD_IDX] * (max_len - len(ids))
    return ids

X_train_ids = [text_to_ids(t) for t in texts_train]
X_valid_ids = [text_to_ids(t) for t in texts_valid]
X_test_ids  = [text_to_ids(t) for t in texts_test]
print('Texts converted to IDs')

In [ ]:
class TextDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = torch.tensor(ids,    dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], self.labels[idx]


class BiLSTM(nn.Module):
    """
    Bidirectional LSTM για text classification.

    Αρχιτεκτονική:
    1. Embedding layer — maps word IDs to dense vectors
    2. BiLSTM — processes sequence in both directions
    3. Pooling — combines forward and backward hidden states
    4. Dropout + Linear — classification head
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 n_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=bidirectional
        )

        # Bidirectional → output size = 2 * hidden_dim
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_output_dim, n_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)

        # LSTM output: (batch, seq_len, hidden_dim * 2)
        lstm_out, (hidden, _) = self.lstm(embedded)

        # Max pooling πάνω στα hidden states
        # Καλύτερο από last hidden state για classification
        pooled = torch.max(lstm_out, dim=1).values  # (batch, hidden_dim * 2)

        return self.classifier(self.dropout(pooled))


def official_st1_score(y_true_h, y_pred_h, y_true_p, y_pred_p, verbose=True):
    f1_h = f1_score(y_true_h, y_pred_h, average='macro', zero_division=0)
    mask = (np.array(y_true_h) == np.array(y_pred_h))
    f1_p = f1_score(
        np.array(y_true_p)[mask], np.array(y_pred_p)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_h + f1_p) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_h:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_p:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

print('BiLSTM architecture defined')

In [ ]:
# Hyperparameters
EMBED_DIM  = 128
HIDDEN_DIM = 256
N_LAYERS   = 2
DROPOUT    = 0.3
BATCH_SIZE = 64
LR         = 1e-3
N_EPOCHS   = 20
PATIENCE   = 3

print('Hyperparameters:')
print(f'  Embedding dim: {EMBED_DIM}')
print(f'  Hidden dim:    {HIDDEN_DIM} (x2 bidirectional = {HIDDEN_DIM*2})')
print(f'  Layers:        {N_LAYERS}')
print(f'  Dropout:       {DROPOUT}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  LR:            {LR}')

In [ ]:
def train_lstm(X_train_ids, y_train, X_valid_ids, y_valid, n_classes, task_name):
    print(f'\n=== ΕΚΠΑΙΔΕΥΣΗ BiLSTM — {task_name} ===')

    train_loader = DataLoader(
        TextDataset(X_train_ids, y_train),
        batch_size=BATCH_SIZE, shuffle=True
    )
    valid_loader = DataLoader(
        TextDataset(X_valid_ids, y_valid),
        batch_size=BATCH_SIZE, shuffle=False
    )

    model = BiLSTM(
        vocab_size=len(vocab),
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        n_classes=n_classes,
        n_layers=N_LAYERS,
        dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=2, factor=0.5, verbose=False
    )
    criterion = nn.CrossEntropyLoss()

    best_f1      = 0
    best_state   = None
    patience_cnt = 0
    history      = []

    for epoch in range(N_EPOCHS):
        # Training
        model.train()
        total_loss = 0
        for ids, labels in train_loader:
            ids, labels = ids.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(ids), labels)
            loss.backward()
            # Gradient clipping — σημαντικό για LSTM
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        all_preds = []
        with torch.no_grad():
            for ids, _ in valid_loader:
                preds = model(ids.to(device)).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)

        f1 = f1_score(y_valid, all_preds, average='macro', zero_division=0)
        history.append({'epoch': epoch+1, 'loss': avg_loss, 'f1': f1})
        scheduler.step(1 - f1)  # Minimize 1-F1

        print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {avg_loss:.4f} | F1: {f1:.4f}', end='')

        if f1 > best_f1:
            best_f1      = f1
            best_state   = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
            print(f' (patience {patience_cnt}/{PATIENCE})')
            if patience_cnt >= PATIENCE:
                print('Early stopping')
                break

    model.load_state_dict(best_state)
    print(f'Καλύτερο F1: {best_f1:.4f}')
    return model, history


def get_predictions(model, ids_list):
    model.eval()
    loader = DataLoader(
        TextDataset(ids_list, [0]*len(ids_list)),
        batch_size=BATCH_SIZE, shuffle=False
    )
    all_preds = []
    with torch.no_grad():
        for ids, _ in loader:
            preds = model(ids.to(device)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

## Εκπαίδευση BiLSTM

In [ ]:
model_hazard, history_h = train_lstm(
    X_train_ids, y_train_hazard,
    X_valid_ids, y_valid_hazard,
    n_hazard, 'HAZARD'
)

In [ ]:
model_product, history_p = train_lstm(
    X_train_ids, y_train_product,
    X_valid_ids, y_valid_product,
    n_product, 'PRODUCT'
)

## Αξιολόγηση & Σύγκριση

In [ ]:
pred_hazard_valid  = le_hazard.inverse_transform(get_predictions(model_hazard,  X_valid_ids))
pred_product_valid = le_product.inverse_transform(get_predictions(model_product, X_valid_ids))

print('=== ΑΞΙΟΛΟΓΗΣΗ BiLSTM (validation) ===')
score_lstm = official_st1_score(
    valid['hazard-category'], pred_hazard_valid,
    valid['product-category'], pred_product_valid
)

print('\n=== ΣΥΓΚΡΙΣΗ NEURAL ARCHITECTURES ===')
print(f'TextCNN:   0.6280  (Kaggle)')
print(f'BiLSTM:    {score_lstm:.4f}  (validation)')
print(f'DistilBERT: 0.7606  (Kaggle)')
print(f'BERT-base:  0.8040  (Kaggle)')

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, history, title in zip(axes,
                               [history_h, history_p],
                               ['Hazard', 'Product']):
    epochs = [h['epoch'] for h in history]
    losses = [h['loss']  for h in history]
    f1s    = [h['f1']    for h in history]

    ax2 = ax.twinx()
    ax.plot(epochs, losses, color='steelblue', label='Loss')
    ax2.plot(epochs, f1s,   color='seagreen',  label='F1', linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss',     color='steelblue')
    ax2.set_ylabel('macro-F1', color='seagreen')
    ax.set_title(f'BiLSTM Learning Curve — {title}')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('lstm_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Αποθηκεύτηκε: lstm_learning_curves.png')

In [ ]:
# Test predictions & submission
test_hazard  = le_hazard.inverse_transform(get_predictions(model_hazard,  X_test_ids))
test_product = le_product.inverse_transform(get_predictions(model_product, X_test_ids))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_lstm.csv', index=False)

print('Αποθηκεύτηκε: submission_lstm.csv')
print(f'Validation score: {score_lstm:.4f}')